# Part 2: Model Training + Confidence-Based Human Review

Trains and compares Random Forest vs Logistic Regression on the engineered
features, using proper stratified 5-fold CV with leakage-safe imputation. Also
implements the fix for the "Partial Match" ambiguity problem: routing
low-confidence predictions to human review instead of forcing a guess.

**Input:** `engineered_features.csv` (from Part 1)
**Output:** `rf_model.joblib`, `model_config.joblib` -- used by Parts 3 and 4

### ⚠️ Known limitation -- read before interpreting results
`match_label` is a deterministic function of `match_score` (hard cutoffs at 0.40
and 0.75, zero exceptions across all 500 rows) -- meaning it's a synthetic proxy
label, not human-annotated ground truth. Our features explain only ~47% of
`match_score`'s variance. We're teaching the model to reconstruct a proxy score,
not "true" recruiter judgment -- worth stating plainly in your report.

## 1. Setup

In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()  # upload engineered_features.csv
except ImportError:
    pass

!pip install scikit-learn -q


In [ ]:
import numpy as np
import pandas as pd
import joblib

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score

RANDOM_STATE = 42


In [ ]:
feature_df = pd.read_csv("engineered_features.csv")
print(feature_df.shape)
feature_df.head(3)


## 2. Prepare features

`experience_match_score` can be NaN when no years-of-experience phrase was found.
We impute it using the **training fold's median only** (never the full dataset's),
to avoid leaking test information into training. We keep `experience_mentioned`
flags so "we don't know" is preserved as its own signal instead of being erased.

In [ ]:
FEATURES = [
    "skill_match_score",
    "matched_skill_count",
    "missing_skill_count",
    "experience_match_score",
    "resume_experience_mentioned",
    "jd_experience_mentioned",
    "semantic_similarity_score",
    "domain_match",
]

X = feature_df[FEATURES].copy()
y = feature_df["match_label"]

print("NaN counts before imputation:")
print(X.isna().sum())


def impute_experience_score(scores, fill_value):
    return scores.fillna(fill_value)


## 3. Random Forest vs Logistic Regression (5-fold CV)

In [ ]:
def run_cv(model_builder, X, y, n_splits=5):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    all_true, all_pred = [], []
    for train_idx, test_idx in skf.split(X, y):
        X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        train_median = X_train["experience_match_score"].median()
        X_train["experience_match_score"] = impute_experience_score(X_train["experience_match_score"], train_median)
        X_test["experience_match_score"] = impute_experience_score(X_test["experience_match_score"], train_median)

        model = model_builder()
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        all_true.extend(y_test)
        all_pred.extend(preds)
    return all_true, all_pred


rf_builder = lambda: RandomForestClassifier(
    n_estimators=300, max_depth=6, min_samples_leaf=5,
    random_state=RANDOM_STATE, class_weight="balanced")

lr_builder = lambda: Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced")),
])

rf_true, rf_pred = run_cv(rf_builder, X, y)
lr_true, lr_pred = run_cv(lr_builder, X, y)

print("=== Random Forest ===")
print(f"Accuracy: {accuracy_score(rf_true, rf_pred):.3f}")
print(classification_report(rf_true, rf_pred))

print("=== Logistic Regression ===")
print(f"Accuracy: {accuracy_score(lr_true, lr_pred):.3f}")
print(classification_report(lr_true, lr_pred))


**Interpreting this:** both models land in a similar place. "Partial Match" has
the weakest recall for both -- not a model-choice problem, but because
`match_label` is `match_score` sliced into bands, and "partial match" sits between
two cutoffs (0.40, 0.75) instead of one.

**We proceed with Random Forest** for compatibility with SHAP `TreeExplainer`
(Part 3) -- not because it's meaningfully more accurate here.

## 4. Confidence-Based Human Review

The real fix for the "Partial Match" problem: use the model's own `predict_proba()`
confidence, and route low-confidence predictions to manual review instead of
forcing a guess -- implementing the "Human-AI Collaborative Decision Support"
feature from the original proposal.

In [ ]:
CONFIDENCE_THRESHOLD = 0.55

def cv_with_confidence(model_builder, X, y, n_splits=5):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    results = []
    for train_idx, test_idx in skf.split(X, y):
        X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        train_median = X_train["experience_match_score"].median()
        X_train["experience_match_score"] = impute_experience_score(X_train["experience_match_score"], train_median)
        X_test["experience_match_score"] = impute_experience_score(X_test["experience_match_score"], train_median)

        model = model_builder()
        model.fit(X_train, y_train)
        proba = model.predict_proba(X_test)
        pred = model.classes_[np.argmax(proba, axis=1)]
        conf = np.max(proba, axis=1)
        for i, idx in enumerate(test_idx):
            results.append({"true": y_test.iloc[i], "pred": pred[i], "confidence": conf[i]})
    return pd.DataFrame(results)


cv_results = cv_with_confidence(rf_builder, X, y)
cv_results["needs_human_review"] = cv_results["confidence"] < CONFIDENCE_THRESHOLD

pct_flagged = cv_results["needs_human_review"].mean() * 100
confident_subset = cv_results[~cv_results["needs_human_review"]]
acc_confident = accuracy_score(confident_subset["true"], confident_subset["pred"])

print(f"Candidates flagged for human review: {pct_flagged:.1f}%")
print(f"Accuracy on confident predictions: {acc_confident:.3f}")
print()
print(classification_report(confident_subset["true"], confident_subset["pred"]))


## 5. Train and save the final production model

In [ ]:
X_full = X.copy()
EXPERIENCE_MEDIAN = X_full["experience_match_score"].median()
X_full["experience_match_score"] = impute_experience_score(X_full["experience_match_score"], EXPERIENCE_MEDIAN)

final_model = RandomForestClassifier(
    n_estimators=300, max_depth=6, min_samples_leaf=5,
    random_state=RANDOM_STATE, class_weight="balanced")
final_model.fit(X_full, y)

joblib.dump(final_model, "rf_model.joblib")
joblib.dump({"experience_median": EXPERIENCE_MEDIAN, "confidence_threshold": CONFIDENCE_THRESHOLD,
             "features": FEATURES}, "model_config.joblib")

print("Saved rf_model.joblib and model_config.joblib -- needed for Parts 3 and 4")

try:
    from google.colab import files
    files.download("rf_model.joblib")
    files.download("model_config.joblib")
except ImportError:
    pass
